In [8]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter

In [9]:
class SHARPTrainDataset(Dataset):
    def __init__(self, root_dir, target_sets=['S1'], window_size=340, step_size=170):
        self.samples =[]
        self.label_map = {'E': 0, 'W': 1, 'R': 2, 'J': 3, 'S': 4, 'C': 5, 'G': 6, 'H': 7}
        
        print(f"Loading training data for sets: {target_sets}...")
        for root, _, files in os.walk(root_dir):
            folder_name = os.path.basename(root)
            
            # Only process folders that start with our target sets (e.g., S1a, S1b match 'S1')
            if not any(folder_name.startswith(ts) for ts in target_sets):
                continue
                
            for file in files:
                if file.endswith(".txt"):
                    try:
                        code = file.split('_')[1][0].upper()
                        if code in self.label_map:
                            label = self.label_map[code]
                            data = np.load(os.path.join(root, file), allow_pickle=True).astype(np.float32)
                            
                            for start in range(0, data.shape[0] - window_size + 1, step_size):
                                window = data[start : start + window_size, :]
                                self.samples.append((window, label))
                    except Exception:
                        continue
        print(f"Loaded {len(self.samples)} train samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        window, label = self.samples[idx]
        return torch.from_numpy(window).unsqueeze(0), torch.tensor(label, dtype=torch.long)

In [10]:
class SHARPTestDataset(Dataset):
    def __init__(self, root_dir, target_sets, window_size=340, step_size=170):
        self.samples =[]
        self.label_map = {'E': 0, 'W': 1, 'R': 2, 'J': 3, 'S': 4, 'C': 5, 'G': 6, 'H': 7}

        for root, _, files in os.walk(root_dir):
            folder_name = os.path.basename(root)
            
            # Only process folders that start with our target sets (e.g., S2a, S2b match 'S2')
            if not any(folder_name.startswith(ts) for ts in target_sets):
                continue

            for file in files:
                if file.endswith("stream_0.txt"): 
                    try:
                        code = file.split('_')[1][0].upper()
                        if code in self.label_map:
                            label = self.label_map[code]
                            base_path = os.path.join(root, file.replace("_stream_0.txt", ""))
                            
                            streams =[]
                            for i in range(4):
                                stream_file = f"{base_path}_stream_{i}.txt"
                                if not os.path.exists(stream_file): 
                                    stream_file = f"{base_path}_stream_0.txt"
                                
                                data = np.load(stream_file, allow_pickle=True).astype(np.float32)
                                streams.append(data)
                            
                            length = streams[0].shape[0]
                            for start in range(0, length - window_size + 1, step_size):
                                end = start + window_size
                                self.samples.append(([s[start:end, :] for s in streams], label))
                    except Exception:
                        continue

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        streams, label = self.samples[idx]
        tensors =[torch.from_numpy(s).unsqueeze(0) for s in streams]
        return tensors[0], tensors[1], tensors[2], tensors[3], torch.tensor(label, dtype=torch.long)

In [11]:
class SHARP_Original_Architecture(nn.Module):
    def __init__(self, num_classes=8, dropout=0.5):
        super().__init__()
        self.branch1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.branch2 = nn.Sequential(nn.Conv2d(1, 5, 2, 2), nn.ReLU())
        self.branch3 = nn.Sequential(
            nn.Conv2d(1, 3, 1, 1), nn.ReLU(),
            nn.ZeroPad2d((0, 1, 0, 1)), 
            nn.Conv2d(3, 6, 2, 1), nn.ReLU(),
            nn.Conv2d(6, 9, 4, 2, 1), nn.ReLU()
        )
        self.concat_conv = nn.Sequential(nn.Conv2d(15, 3, 1, 1), nn.ReLU())
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(dropout)
        self.dense = nn.Linear(25500, num_classes)

    def forward(self, x):
        x = torch.cat([self.branch1(x), self.branch2(x), self.branch3(x)], dim=1)
        x = self.concat_conv(x)
        x = self.dense(self.dropout(self.flatten(x)))
        return x

In [ ]:
def sharp_decision_fusion(logits_list):
    stacked_logits = torch.stack(logits_list) 
    stacked_preds = torch.argmax(stacked_logits, dim=2) 
    final_preds =[]
    
    for i in range(stacked_preds.shape[1]):
        preds_for_sample = stacked_preds[:, i].tolist()
        counts = Counter(preds_for_sample)
        most_common_pred, count = counts.most_common(1)[0]
        
        if count >= 3:
            final_preds.append(most_common_pred)
        else:
            summed_logits = torch.sum(stacked_logits[:, i, :], dim=0)
            final_preds.append(torch.argmax(summed_logits).item())
            
    return torch.tensor(final_preds, device=logits_list[0].device)

In [13]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running on: {device}")

    # ==========================================
    # 1. LOAD TRAINING DATA
    # ==========================================
    # Pass the root directory and tell it to find ALL folders starting with 'S1' (S1a, S1b, S1c)
    train_dataset = SHARPTrainDataset("doppler_traces", target_sets=['S1']) 
    
    # Split train/val (75% for train, 25% for val)
    train_size = int(0.75 * len(train_dataset)) 
    val_size = len(train_dataset) - train_size
    train_set, val_set = torch.utils.data.random_split(train_dataset, [train_size, val_size])

    train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=128, shuffle=False)
    
# ==========================================
    # 2. INIT MODEL, OPTIMIZER & SCHEDULER
    # ==========================================
    # INCREASED DROPOUT TO 0.5 TO PREVENT OVERFITTING
    model = SHARP_Original_Architecture(num_classes=8, dropout=0.5).to(device)
    criterion = nn.CrossEntropyLoss()
    
    # ADDED WEIGHT DECAY (L2 Regularization)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    # ADDED SCHEDULER (Smoothly lowers the learning rate over 30 epochs)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

    # ==========================================
    # 3. TRAINING LOOP
    # ==========================================
    best_val_acc = 0.0
    epochs = 30
    
    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            train_total += targets.size(0)
            train_correct += (predicted == targets).sum().item()

        # Step the scheduler at the end of the epoch
        scheduler.step()

        # Validation Loop
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += targets.size(0)
                val_correct += (predicted == targets).sum().item()

        val_acc = val_correct / val_total
        
        # Get current learning rate to print it
        current_lr = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch+1}/{epochs}[LR: {current_lr:.5f}] | Train Acc: {train_correct/train_total:.4f} | Val Acc: {val_acc:.4f}")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "sharp_best_weights.pth")

    # ==========================================
    # 4. TESTING LOOP (EVALUATING S2 TO S7)
    # ==========================================
    print("\n--- Starting SHARP Decision Fusion Testing ---")
    
    # Load the best weights we just trained on S1
    model.load_state_dict(torch.load("sharp_best_weights.pth"))
    model.eval()
    
    # We want to test on sets S2 through S7
    test_sets =['S2', 'S3', 'S4', 'S5', 'S6', 'S7']
    
    # A dictionary to store results to print a nice table at the end
    results = {}

    with torch.no_grad():
        for test_set_id in test_sets:
            print(f"\nEvaluating {test_set_id} (combining all sub-campaigns)...")
            
            # Load the specific test set using target_sets
            test_dataset = SHARPTestDataset("doppler_traces", target_sets=[test_set_id])
            
            # Skip if dataset wasn't found or is empty
            if len(test_dataset) == 0:
                print(f"  -> No data found for {test_set_id}. Skipping.")
                continue
                
            test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
            
            test_correct = 0
            test_total = 0
            
            # Run the specific test set through the model
            for x1, x2, x3, x4, targets in test_loader:
                x1, x2, x3, x4 = x1.to(device), x2.to(device), x3.to(device), x4.to(device)
                targets = targets.to(device)
                
                # Predict independently on each antenna stream
                l1, l2, l3, l4 = model(x1), model(x2), model(x3), model(x4)
                
                # Apply Paper Section 4.2 Fusion
                final_preds = sharp_decision_fusion([l1, l2, l3, l4])
                
                test_total += targets.size(0)
                test_correct += (final_preds == targets).sum().item()
            
            # Calculate and store accuracy for this set
            accuracy = test_correct / test_total
            results[test_set_id] = accuracy
            print(f"  -> Accuracy for {test_set_id}: {accuracy * 100:.2f}%")

    # ==========================================
    # 5. PRINT FINAL REPORT (Like Table 3)
    # ==========================================
    print("\n" + "="*40)
    print(" SHARP GENERALIZATION REPORT (TABLE 3)")
    print("="*40)
    print(f" Trained on: S1 (Combines S1a, S1b, S1c)")
    print("-" * 40)
    for folder, acc in results.items():
        print(f" Test Set {folder} Accuracy : {acc * 100:.2f}%")
    print("="*40)

Running on: cuda
Loading training data for sets: ['S1']...
Loaded 9100 train samples.
Epoch 1/30[LR: 0.00100] | Train Acc: 0.3029 | Val Acc: 0.3626
Epoch 2/30[LR: 0.00099] | Train Acc: 0.4097 | Val Acc: 0.3851
Epoch 3/30[LR: 0.00098] | Train Acc: 0.4349 | Val Acc: 0.4211
Epoch 4/30[LR: 0.00096] | Train Acc: 0.4579 | Val Acc: 0.4668
Epoch 5/30[LR: 0.00093] | Train Acc: 0.4633 | Val Acc: 0.4765
Epoch 6/30[LR: 0.00090] | Train Acc: 0.4637 | Val Acc: 0.4659
Epoch 7/30[LR: 0.00087] | Train Acc: 0.4693 | Val Acc: 0.4277
Epoch 8/30[LR: 0.00083] | Train Acc: 0.4719 | Val Acc: 0.4756
Epoch 9/30[LR: 0.00079] | Train Acc: 0.4881 | Val Acc: 0.4901
Epoch 10/30[LR: 0.00075] | Train Acc: 0.5032 | Val Acc: 0.4273
Epoch 11/30[LR: 0.00070] | Train Acc: 0.4976 | Val Acc: 0.4866
Epoch 12/30[LR: 0.00065] | Train Acc: 0.5053 | Val Acc: 0.4668
Epoch 13/30[LR: 0.00060] | Train Acc: 0.5090 | Val Acc: 0.5024
Epoch 14/30[LR: 0.00055] | Train Acc: 0.5166 | Val Acc: 0.4651
Epoch 15/30[LR: 0.00050] | Train Acc: 0.5